---
# 🦺 Detección de EPP con YOLOv8
### Proyecto de Visión por Computadora
**Autor:** Juan José Vargas  
**Dataset:** PPE Factory — Roboflow  
**Modelo:** YOLOv8s · 50 épocas · GPU Tesla T4

---
> **Paso 1 — Instalación de dependencias**  
> Se instalan `ultralytics` (YOLOv8) y `roboflow` (acceso al dataset). El flag `--quiet` suprime la salida detallada de pip.

In [1]:
 !pip install ultralytics roboflow --quiet  # Instala YOLOv8 (ultralytics) y el cliente de Roboflow para descargar datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 140.3 MB/s eta 0:00:00


## 📦 Paso 2 — Descarga del Dataset desde Roboflow

| Parámetro | Valor |
|---|---|
| Workspace | `efrains-workspace-5kbud` |
| Proyecto | `ppe-factory-bmdcj-uiafg` |
| Versión | `1` |
| Formato | `YOLOv8` |

Se inicializa el cliente de Roboflow con la API key y se descarga el dataset completo en formato compatible con YOLOv8.

In [2]:
from roboflow import Roboflow

# --- Credenciales de Roboflow ---
ROBOFLOW_API_KEY = "jxFTJNrLtq44GCx62i9e"  # Llave de acceso a la API de Roboflow
rfc = Roboflow(api_key=ROBOFLOW_API_KEY)      # Inicializa el cliente de Roboflow
print("Successfully initialized Roboflow client!")

# --- Parámetros del dataset ---
project_name   = "ppe-factory-bmdcj-uiafg"   # Nombre del proyecto en Roboflow
version_number = 1                           # Versión del dataset a descargar
workspace_name = "efrains-workspace-5kbud"   # Workspace extraído del mensaje de error previo

# --- Descarga del dataset en formato YOLOv8 ---
try:
    project = rfc.workspace(workspace_name).project(project_name)
    dataset = project.version(version_number).download("yolov8")  # También acepta "coco", "pascal", etc.
    print(f"Dataset '{project_name}' version {version_number} downloaded successfully!")
    print(f"Dataset path: {dataset.location}")
except Exception as e:
    print(f"Error downloading dataset: {e}")
    print("Please ensure your API key, project name, workspace name, and version number are correct.")

Successfully initialized Roboflow client!
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ppe-factory-1 in yolov8:: 100%|██████████| 11037/11037 [00:01<00:00, 6494.88it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset 'ppe-factory-bmdcj-uiafg' version 1 downloaded successfully!
Dataset path: /content/ppe-factory-1


## 🗂️ Paso 3 — Exploración de la Estructura del Dataset

Se examina el directorio del dataset para verificar que los tres *splits* estén correctamente organizados:

- 🟢 **train/** → imágenes y etiquetas para entrenamiento  
- 🟡 **valid/** → imágenes y etiquetas para validación  
- 🔵 **test/** → imágenes para evaluación final

La función `count_files_in_directory` recorre recursivamente cada carpeta y cuenta el total de archivos.

In [3]:
import os

dataset_path = '/content/ppe-factory-1'  # Ruta raíz del dataset descargado

print(f"Exploring dataset directory: {dataset_path}\n")

# Función auxiliar: cuenta archivos de forma recursiva dentro de un directorio
def count_files_in_directory(path):
    count = 0
    for root, dirs, files in os.walk(path):  # Recorre subdirectorios
        count += len(files)
    return count

# Muestra los subdirectorios de primer nivel
print("Top-level directories:")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        print(f"- {item}/")

# Conteo de archivos por split: train, valid y test
print("\nDetailed file counts per split:")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset_path, split)
    if os.path.isdir(split_path):
        num_files = count_files_in_directory(split_path)
        print(f"- {split}/: {num_files} files")
    else:
        print(f"- {split}/: Not found or not a directory")

# Total general de archivos en el dataset
print("\nOverall file count in dataset directory:")
overall_files = count_files_in_directory(dataset_path)
print(f"Total files in '{dataset_path}': {overall_files}")

Exploring dataset directory: /content/ppe-factory-1

Top-level directories:
- train/
- valid/
- test/

Detailed file counts per split:
- train/: 10004 files
- valid/: 742 files
- test/: 286 files

Overall file count in dataset directory:
Total files in '/content/ppe-factory-1': 11035


## 🏷️ Paso 4 — Lectura de Clases desde `data.yaml`

> **¿Qué es `data.yaml`?**  
> Archivo de configuración que define las rutas de cada split y el listado de clases del dataset.

El modelo reconocerá **7 categorías de EPP**:

`boots` · `earmuffs` · `glasses` · `gloves` · `helmet` · `person` · `vest`

In [4]:
import yaml

data_yaml_path = '/content/ppe-factory-1/data.yaml'  # Ruta al archivo de configuración del dataset

try:
    # Abre y carga el archivo YAML de forma segura
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Imprime las etiquetas (clases) definidas en el dataset
    if 'names' in data:
        print("Labels (names) in data.yaml:")
        for i, name in enumerate(data['names']):
            print(f"- {i}: {name}")  # Índice numérico y nombre de la clase
    else:
        print("The 'names' key (labels) was not found in data.yaml.")

    # Muestra el contenido completo del archivo
    print("\nFull content of data.yaml:")
    print(yaml.dump(data, indent=2))

except FileNotFoundError:
    print(f"Error: data.yaml not found at {data_yaml_path}")
except yaml.YAMLError as e:
    print(f"Error parsing data.yaml: {e}")

Labels (names) in data.yaml:
- 0: boots
- 1: earmuffs
- 2: glasses
- 3: gloves
- 4: helmet
- 5: person
- 6: vest

Full content of data.yaml:
names:
- boots
- earmuffs
- glasses
- gloves
- helmet
- person
- vest
nc: 7
roboflow:
  license: CC BY 4.0
  project: ppe-factory-bmdcj-uiafg
  url: https://universe.roboflow.com/efrains-workspace-5kbud/ppe-factory-bmdcj-uiafg/dataset/1
  version: 1
  workspace: efrains-workspace-5kbud
test: ../test/images
train: ../train/images
val: ../valid/images



---
## 🤖 Paso 5 — Carga del Modelo Base YOLOv8s

Se utiliza **YOLOv8s** (*small*) como punto de partida mediante *transfer learning*:  
los pesos preentrenados en COCO se ajustarán al dominio de detección de EPP.

```
YOLOv8s → 11.1 M parámetros · 28.8 GFLOPs · 129 capas
```
---

In [5]:
from ultralytics import YOLO

# Carga el modelo base YOLOv8s preentrenado en COCO (transfer learning)
# 's' = small: buen equilibrio entre velocidad y precisión
model = YOLO('yolov8s.pt')

# Muestra un resumen de la arquitectura: capas, parámetros y GFLOPs
model.info()

YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.816844800000002)

## 🏋️ Paso 6 — Entrenamiento del Modelo

### Hiperparámetros de entrenamiento

| Parámetro | Valor | Descripción |
|---|---|---|
| `epochs` | 50 | Pasadas completas sobre el dataset |
| `imgsz` | 416 | Resolución de entrada (px) |
| `batch` | 16 | Imágenes por iteración |
| `optimizer` | AdamW (auto) | Seleccionado automáticamente por Ultralytics |

Los resultados se guardan en `/content/runs/ppe_detector/`.

In [6]:
# Entrena el modelo con los hiperparámetros definidos
results = model.train(
    data="/content/ppe-factory-1/data.yaml",  # Ruta al archivo de configuración del dataset
    epochs=50,                                 # Número total de épocas de entrenamiento
    imgsz=416,                                 # Tamaño de entrada de imagen (px)
    batch=16,                                  # Imágenes por lote (ajustar según VRAM disponible)
    name="ppe_detector",                       # Nombre del experimento (carpeta de salida)
    project="/content/runs"                    # Directorio raíz donde se guardan los resultados
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ppe-factory-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

---
### ✅ Paso 7 — Carga del Mejor Modelo

Después del entrenamiento, `best.pt` contiene los pesos que lograron el mayor **mAP50** en el conjunto de validación. Este archivo es el que se usará para inferencia y exportación.

In [9]:
from ultralytics import YOLO

# Ruta a los pesos del mejor modelo guardado durante el entrenamiento
# 'best.pt' se selecciona automáticamente según la métrica mAP50 en validación
best_model_path = '/content/runs/ppe_detector/weights/best.pt'
model = YOLO(best_model_path)  # Carga el modelo con los mejores pesos

print(f"Best model loaded successfully from: {best_model_path}")

Best model loaded successfully from: /content/runs/ppe_detector/weights/best.pt


## 📊 Paso 8 — Evaluación del Modelo

Se calculan las métricas estándar de detección de objetos sobre el conjunto de **validación**:

| Métrica | Descripción |
|---|---|
| **mAP50** | mAP con umbral IoU = 0.50 |
| **mAP50-95** | mAP promediado en IoU 0.50 → 0.95 |
| **Precisión** | TP / (TP + FP) |
| **Recall** | TP / (TP + FN) |

In [10]:
# Ejecuta la validación sobre el conjunto de datos de validación
# Requiere que 'data_yaml_path' esté definido en una celda anterior
metrics = model.val(data=data_yaml_path)

# Imprime las métricas de evaluación del modelo
print("\n📊 Métricas de evaluación:")
print(f"  mAP50:    {metrics.box.map50:.4f}")  # mAP a umbral IoU = 0.50
print(f"  mAP50-95: {metrics.box.map:.4f}")    # mAP promediado en IoU 0.50–0.95
print(f"  Precisión: {metrics.box.mp:.4f}")    # Precisión media entre clases
print(f"  Recall:    {metrics.box.mr:.4f}")    # Recall medio entre clases

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,128,293 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1512.0±961.0 MB/s, size: 58.0 KB)
val: Scanning /content/ppe-factory-1/valid/labels.cache... 371 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 371/371 129.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 4.2it/s 5.7s
                   all        371       1914      0.806       0.71      0.788      0.529
                 boots        144        326      0.958      0.763      0.887      0.612
              earmuffs        109        172      0.746      0.616      0.673      0.426
               glasses        111        132      0.842      0.608       0.75       0.38
                gloves         75        155      0.411       0.51       0.51      0.301
                helmet        241       

## 🖼️ Paso 9 — Inferencia en Imagen Aleatoria de Test

Se selecciona una imagen al azar del conjunto de *test* y se ejecuta la inferencia.  
El resultado se guarda automáticamente en la carpeta `runs/detect/predict/` con los bounding boxes dibujados sobre la imagen original.

> 💡 El umbral `conf=0.25` filtra detecciones con menos del 25 % de confianza.

In [11]:
import random
import os
import glob
from IPython.display import Image, display

# Directorio con las imágenes de prueba
test_images_path = '/content/ppe-factory-1/test/images'

# Recolecta todas las imágenes disponibles (jpg, jpeg, png)
image_files = glob.glob(os.path.join(test_images_path, '*.jpg')) + \
              glob.glob(os.path.join(test_images_path, '*.jpeg')) + \
              glob.glob(os.path.join(test_images_path, '*.png'))

if not image_files:
    print(f"No se encontraron imágenes en: {test_images_path}")
else:
    # Selecciona una imagen al azar para probar el modelo
    random_image = random.choice(image_files)
    print(f"Probando el modelo con la imagen: {random_image}")

    # Ejecuta la inferencia y guarda la imagen anotada
    results = model.predict(source=random_image, save=True, conf=0.25)  # conf=0.25 umbral mínimo de confianza

    # Construye la ruta a la imagen de predicción guardada por YOLOv8
    runs_dir = os.path.join(model.trainer.save_dir, 'predict') if hasattr(model, 'trainer') and model.trainer else os.path.join('/content/runs/detect', 'predict')
    latest_predict_run = sorted(glob.glob(os.path.join(runs_dir, '*')))[-1]  # Carpeta más reciente
    predicted_image_path = os.path.join(latest_predict_run, os.path.basename(random_image))

    # Muestra la imagen con los bounding boxes si existe
    if os.path.exists(predicted_image_path):
        display(Image(filename=predicted_image_path))
    else:
        print("No se encontró la imagen predicha. Asegúrate de que 'save=True' se haya ejecutado correctamente y la ruta sea correcta.")


Probando el modelo con la imagen: /content/ppe-factory-1/test/images/images399_jpg.rf.1a230972abae42402d3ba6298a864edd.jpg

image 1/1 /content/ppe-factory-1/test/images/images399_jpg.rf.1a230972abae42402d3ba6298a864edd.jpg: 288x416 2 earmuffss, 2 glassess, 2 helmets, 2 persons, 46.0ms
Speed: 0.9ms preprocess, 46.0ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 416)
Results saved to /content/runs/detect/predict
No se encontró la imagen predicha. Asegúrate de que 'save=True' se haya ejecutado correctamente y la ruta sea correcta.


## 🔁 Paso 10 — Inferencia sobre Todo el Conjunto de Test

Se procesa cada una de las **143 imágenes** de test de forma secuencial.  
Para cada imagen se registra el número de detecciones obtenidas, permitiendo identificar casos con cero detecciones o cantidad inusualmente alta.

> ⚡ Se usa `verbose=False` para reducir el ruido en la salida del bucle.

In [12]:
# Predicción en TODAS las imágenes del conjunto de test
import os
from ultralytics import YOLO

# Directorio corregido con las imágenes de prueba
test_images_dir = "/content/ppe-factory-1/test/images"

# Filtra únicamente archivos de imagen válidos
all_images = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"🔍 Procesando {len(all_images)} imágenes de test...")
print("-" * 50)

# Itera sobre cada imagen y ejecuta la inferencia
for i, filename in enumerate(all_images):
    image_path = os.path.join(test_images_dir, filename)
    results = model.predict(image_path, save=True, imgsz=640, conf=0.25, verbose=False)  # verbose=False silencia logs por imagen
    n_detections = len(results[0].boxes)  # Número de objetos detectados
    print(f"[{i+1:03d}/{len(all_images)}] {filename:<50} → {n_detections} detecciones")

print("-" * 50)
print("✅ Predicciones completadas. Resultados guardados en /content/runs/")

🔍 Procesando 143 imágenes de test...
--------------------------------------------------
Results saved to /content/runs/detect/predict
[001/143] ear-protection-4-_jpg.rf.e3de4dc016a170a1fd64223cff8c6461.jpg → 2 detecciones
Results saved to /content/runs/detect/predict
[002/143] image_197_jpg.rf.7b1234e105550ce79b22ad1675d8b1fe.jpg → 7 detecciones
Results saved to /content/runs/detect/predict
[003/143] Video4_4_jpg.rf.02cf1779d5ee5df5a7c46ae85611667a.jpg → 7 detecciones
Results saved to /content/runs/detect/predict
[004/143] outputFile3_11_png.rf.cc1dddc6e21d6027bc600a9968ab71a1.jpg → 13 detecciones
Results saved to /content/runs/detect/predict
[005/143] IMG_6499-e1439982204742_jpg.rf.5609f8d9c0faea4b64212f73ccdf61fe.jpg → 26 detecciones
Results saved to /content/runs/detect/predict
[006/143] gozluk_055_jpg.rf.c05e90d4cc21483c4001efe43558a041.jpg → 2 detecciones
Results saved to /content/runs/detect/predict
[007/143] yt-hAWOY80H22M-0094_jpg.rf.a4cf0e738a5b01d1bc58f20f50426eb6.jpg → 3 det

---
## 🎬 Paso 11 — Descarga del Video de Ejemplo

Se descarga un video de ejemplo desde GitHub usando `wget`.  
Este video se usará en el siguiente paso para probar la detección en secuencias de video.

```
Fuente: github.com/adiacla/ppe · Tamaño: ~31 MB
```
---

In [14]:
# Descarga el video de ejemplo directamente desde el repositorio de GitHub
# El flag -O indica la ruta de destino en el entorno de Colab
!wget -O /content/example_video.mp4 https://github.com/adiacla/ppe/raw/main/video/example_video.mp4


--2026-04-24 20:12:22--  https://github.com/adiacla/ppe/raw/main/video/example_video.mp4
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adiacla/ppe/main/video/example_video.mp4 [following]
--2026-04-24 20:12:22--  https://raw.githubusercontent.com/adiacla/ppe/main/video/example_video.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32903392 (31M) [application/octet-stream]
Saving to: ‘/content/example_video.mp4’

/content/example_vi 100%[===================>]  31.38M  --.-KB/s    in 0.1s    

2026-04-24 20:12:24 (273 MB/s) - ‘/content/example_video.mp4’ saved [32903392/32903392]



## 🎥 Paso 12 — Detección en Video

Se aplica el modelo entrenado sobre un video completo, frame a frame.  
Los resultados (video anotado) se guardan en `/content/runs/video_test/`.

> ⚠️ **Nota:** Para videos largos se recomienda usar `stream=True` para evitar errores de memoria RAM.

In [16]:
# Predicción en video completo
# Ajusta 'video_source' a la ruta de tu video (puede estar en /content/ o en Google Drive)

video_source = "/content/example_video.mp4"  # Ruta del video descargado desde GitHub

# Nota: Si el modelo no está cargado, ejecuta primero:
# model = YOLO("/content/runs/ppe_detector/weights/best.pt")

if os.path.exists(video_source):
    results = model.predict(
        source=video_source,          # Archivo de video de entrada
        conf=0.3,                     # Umbral de confianza (más alto = menos falsos positivos)
        save=True,                    # Guarda el video con las detecciones dibujadas
        project="/content/runs",      # Carpeta base para guardar resultados
        name="video_test"             # Subcarpeta específica del experimento
    )
    print("✅ Predicción en video completada. Resultados guardados en /content/runs/video_test/")
else:
    print(f"⚠️ No se encontró el video en: {video_source}")
    print("Asegúrese de haber descargado el video y que la ruta sea correcta.")

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to /content/runs/video_test
✅ Predicción en video completada. Resultados guardados en /content/runs/video_test/


---
## 💾 Paso 13 — Exportación del Modelo a ONNX

**ONNX** (Open Neural Network Exchange) es un formato abierto para interoperabilidad entre frameworks.  
El modelo exportado puede usarse con:

- 🚀 **ONNX Runtime** — inferencia multiplataforma  
- ⚡ **TensorRT** — optimización en GPU NVIDIA  
- 🔵 **OpenVINO** — hardware Intel

**Juan José Vargas** — Proyecto EPP Detection 🦺
---

In [15]:
# Exporta el modelo entrenado al formato ONNX para despliegue en producción
# ONNX es compatible con múltiples frameworks: TensorRT, OpenVINO, ONNX Runtime, etc.
export_path = model.export(format='onnx')  # Genera best.onnx en la misma carpeta de pesos

print(f"✅ Modelo exportado exitosamente a: {export_path}")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/ppe_detector/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 11, 3549) (21.4 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 373ms
Prepared 4 packages in 9.29s
Installed 4 packages in 342ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.25.0
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 10.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 12.6s, saved as '/content/r